In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import spacy
import glob
from constants import BASE_PATH
import json
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import text_cleaning as tc
from collections import defaultdict
import seaborn as sns
import matplotlib.pyplot as plt
import pickle

In [9]:
from constants import (
    JIBARO_WORDS, GIBARO_WORDS, CAMPESINO_WORDS, 
    LCCN_TO_NAME
)

In [4]:
nlp = spacy.blank("es")

In [5]:
json_paths  = glob.glob(f'{BASE_PATH}**/cleaned.json', recursive=True)

In [6]:
json_paths[0]

'data/sn86077151/1921/05/30/ed-1/seq-8/cleaned.json'

In [7]:
len(json_paths)

163765

In [10]:
target_words = set(JIBARO_WORDS + GIBARO_WORDS + CAMPESINO_WORDS)

In [11]:
target_words


{'campesina',
 'campesinas',
 'campesinillas',
 'campesinillos',
 'campesinita',
 'campesinitas',
 'campesinito',
 'campesinitos',
 'campesino',
 'campesinos',
 'gibara',
 'gibaras',
 'gibaritos',
 'gibaro',
 'gibaros',
 'gíbara',
 'gíbaras',
 'gíbaritos',
 'gíbaro',
 'gíbaros',
 'jibara',
 'jibaras',
 'jibaritos',
 'jibaro',
 'jibaros',
 'jíbara',
 'jíbaras',
 'jíbaritos',
 'jíbaro',
 'jíbaros'}

In [10]:
word_index = defaultdict(list)
text_block_map = {}
paper_data = []

keys_to_copy = ['lccn', 'date', 'year', 'month', 'day', 'edition', 'sequence', 'link']
for path in tqdm(json_paths):
    try:
        with open(path, 'r') as f:
            d = json.load(f)
            dd = {k: d[k] for k in keys_to_copy}
            total_tokens = 0
            for i, tb in enumerate(d['text_blocks_cleaned']):
                text_block_map[(i, d['link'])] = tb['text']
                for tok in nlp(tb['text']):
                    total_tokens += 1 
                    if tok.text.lower() in target_words:
                        word_index[tok.text.lower()].append((i, d['link']))
            dd['total_tokens'] = total_tokens
            paper_data.append(dd)
    except json.JSONDecodeError as e:
        # Handle the specific error if decoding fails
        print("Error: Unable to decode JSON.")
        print("Details:", e)


100%|██████████████████████████████████████████████| 163765/163765 [39:43<00:00, 68.71it/s]


In [11]:
len(paper_data)

163765

In [12]:
dataset_df = pd.DataFrame.from_dict(paper_data)
dataset_df['title'] = dataset_df['lccn'].map(LCCN_TO_NAME)

In [13]:
dataset_df = dataset_df.set_index('link')

In [14]:
dataset_df.to_pickle('prelim_analysis_df.pkl')

In [15]:
with open('word_index.pkl', 'wb') as f:
    pickle.dump(word_index, f)

In [16]:
jibaro_counts = defaultdict(int)
gibaro_counts = defaultdict(int)
campsino_counts = defaultdict(int)
for w in JIBARO_WORDS:
    for _, l in word_index[w]:
        jibaro_counts[l] += 1

for w in GIBARO_WORDS:
    for _, l in word_index[w]:
        gibaro_counts[l] += 1

for w in CAMPESINO_WORDS:
    for _, l in word_index[w]:
        campsino_counts[l] += 1

In [17]:
dataset_df['jibaro_counts'] = [jibaro_counts[i] for i in dataset_df.index]
dataset_df['gibaro_counts'] = [gibaro_counts[i] for i in dataset_df.index]
dataset_df['campsino_counts'] = [campsino_counts[i] for i in dataset_df.index]

dataset_df['contains_search_word_jibar'] = dataset_df['jibaro_counts'] > 0
dataset_df['contains_search_word_gibar'] = dataset_df['gibaro_counts'] > 0
dataset_df['contains_search_word_campesin'] = dataset_df['campsino_counts'] > 0

dataset_df['jibaro_freq'] = dataset_df['jibaro_counts'] / dataset_df['total_tokens']
dataset_df['gibaro_freq'] = dataset_df['gibaro_counts'] / dataset_df['total_tokens']
dataset_df['campesino_freq'] = dataset_df['campsino_counts'] / dataset_df['total_tokens']

In [18]:
dataset_df[dataset_df[['jibaro_counts', 'gibaro_counts', 'campsino_counts']].sum(axis=1) > 0].to_pickle('prelim_analysis_target_records_df.pkl')
